# 🌍 Notebook 3 — Real-world Service Discovery

So far we've built everything from scratch to understand the pattern. In real systems you
almost never write a registry yourself — you pick one off the shelf.

This notebook walks through the four options you'll meet on the job:

1. **Kubernetes DNS + Services** — server-side discovery baked into the platform.
2. **HashiCorp Consul** — standalone registry with heartbeats and HTTP API.
3. **Netflix Eureka** — the classic client-side discovery registry (Spring Cloud world).
4. **Service Mesh (Istio / Linkerd / Envoy)** — discovery + routing handled by a sidecar.

For each we look at *how a service is registered*, *how a client looks it up*, and
*what you'd actually type*.


## 🛠️ Setup

```bash
cd 05-microservices/service-discovery
uv sync
```

Select the `.venv` kernel. Reload VS Code if it doesn't appear.

> This notebook is mostly conceptual — the real systems need clusters to run. We include
> one runnable Python simulation of **Kubernetes-style DNS** so you can *feel* how clients
> experience it.


## 0. The simplest thing that works: plain DNS

Before reaching for Consul or a mesh, remember that **DNS itself is a service registry**.
Put multiple `A` records behind a hostname (or let your cloud load balancer do it) and
clients get round-robin for free:

```
$ dig +short orders.internal
10.0.0.1
10.0.0.2
10.0.0.3
```

```python
import socket
print(socket.gethostbyname_ex("orders.internal"))
# ('orders.internal', [], ['10.0.0.1', '10.0.0.2', '10.0.0.3'])
```

**Why it's tempting**

- Zero new infrastructure. Every language already speaks DNS.
- Works fine for small, stable fleets.

**Why teams outgrow it**

- DNS TTLs + OS/JVM caches make changes slow to propagate (minutes, not seconds).
- No health awareness — dead IPs stay in the record until something removes them.
- No richer routing (weighting, canaries, retries).

Kubernetes Services are essentially *"DNS, but with a controller that keeps the record
correct in near real time."* That's the upgrade path.


## 1. Kubernetes DNS + Services (server-side)

Kubernetes is the most common place you'll see service discovery today. You don't install
anything — the platform does it.

**How it works**

- You deploy pods (instances) with a label like `app=orders`.
- You declare a **Service** object matching those pods. K8s assigns it a stable virtual IP
  (ClusterIP) and a DNS name, e.g. `orders.default.svc.cluster.local`.
- Kubernetes' endpoints controller maintains the list of healthy pod IPs behind that Service
  (driven by *readiness probes* — pull-style health checks).
- A client calls `http://orders/` — DNS resolves to the Service IP, `kube-proxy` load-balances
  to a healthy pod. **Pure server-side discovery, zero client code.**

```yaml
# Service manifest (abridged)
apiVersion: v1
kind: Service
metadata: { name: orders }
spec:
  selector: { app: orders }
  ports: [{ port: 80, targetPort: 8080 }]
```

```yaml
# Deployment with a readiness probe — the "pull" health check
apiVersion: apps/v1
kind: Deployment
metadata: { name: orders }
spec:
  replicas: 3
  selector: { matchLabels: { app: orders } }
  template:
    metadata: { labels: { app: orders } }
    spec:
      containers:
        - name: orders
          image: myrepo/orders:1.2.0
          readinessProbe:
            httpGet: { path: /healthz, port: 8080 }
            periodSeconds: 5
```

From inside the cluster a client just writes:

```python
requests.get("http://orders/v1/orders/42")
```

…and Kubernetes handles the discovery. This is the canonical modern setup.


### A tiny Python simulation of the Kubernetes model

We'll simulate DNS, a Service virtual IP, and `kube-proxy`'s round-robin forwarding.


In [ ]:
import itertools

class KubeDNS:
    def __init__(self):
        self._records = {}  # service_name -> "virtual IP"
    def register_service(self, name, vip):
        self._records[name] = vip
    def resolve(self, name):
        return self._records[name]

class KubeService:
    """Holds pod IPs that pass the readiness probe; round-robins between them."""
    def __init__(self, vip, readiness_probe):
        self.vip = vip
        self.pods = []
        self.probe = readiness_probe
        self._rr = itertools.count()

    def update_endpoints(self, all_pod_ips):
        # Like the endpoints controller: keep pods that currently pass the probe.
        self.pods = [ip for ip in all_pod_ips if self.probe(ip)]

    def proxy(self, path):
        if not self.pods:
            return {"status": 503, "error": "no ready pods"}
        ip = self.pods[next(self._rr) % len(self.pods)]
        return {"status": 200, "routed_to": ip, "path": path}


dns = KubeDNS()
orders_svc = KubeService("10.96.0.10", readiness_probe=lambda ip: ip != "10.1.0.2")
dns.register_service("orders", "10.96.0.10")

# Pretend 3 pods exist; one is NOT ready.
all_pods = ["10.1.0.1", "10.1.0.2", "10.1.0.3"]
orders_svc.update_endpoints(all_pods)

# Client side: just resolve the name and call the VIP.
vip = dns.resolve("orders")
print(f"orders resolves to {vip} (the Service IP)")
for _ in range(4):
    print(orders_svc.proxy("/v1/orders/42"))


## 2. HashiCorp Consul (self-reg + HTTP API)

Consul is a popular standalone registry. Each service registers itself (or via a Consul
agent running on the same node) and can declare heartbeat-based or HTTP-based health checks.

**Register via HTTP API** (sketch):

```bash
curl -X PUT http://consul:8500/v1/agent/service/register -d '{
  "Name": "orders",
  "ID":   "orders-1",
  "Address": "10.0.0.1",
  "Port": 8080,
  "Check": {
    "HTTP": "http://10.0.0.1:8080/health",
    "Interval": "5s"
  }
}'
```

**Look up healthy instances**:

```bash
curl http://consul:8500/v1/health/service/orders?passing=true
```

You'd normally use a client library rather than raw HTTP, but the shape is:
*self-registration + pull-style HTTP health checks + client-side discovery.*


In [ ]:
# Minimal Python sketch — what your "use Consul" code might look like if you squint.
# We keep the request layer fake so this notebook runs anywhere.

class FakeConsul:
    def __init__(self):
        self.services = {}
    def register(self, name, sid, addr, port, health_fn):
        self.services.setdefault(name, {})[sid] = (addr, port, health_fn)
    def healthy(self, name):
        out = []
        for sid, (addr, port, fn) in self.services.get(name, {}).items():
            if fn():
                out.append(f"{addr}:{port}")
        return out

consul = FakeConsul()
consul.register("orders", "orders-1", "10.0.0.1", 8080, health_fn=lambda: True)
consul.register("orders", "orders-2", "10.0.0.2", 8080, health_fn=lambda: False)  # unhealthy
print("passing:", consul.healthy("orders"))


## 3. Netflix Eureka (client-side discovery)

Eureka is the classic **client-side discovery** registry — the one Netflix open-sourced
and Spring Cloud made famous. Each service instance runs an embedded *Eureka client* that:

1. Registers with the Eureka server on startup.
2. Sends a heartbeat every 30 s by default.
3. Caches the full registry locally and refreshes it every 30 s.
4. Uses a client-side load balancer (Ribbon / Spring Cloud LoadBalancer) to pick an instance
   from that cache on every outbound call.

Typical Spring flow:

```java
@FeignClient("orders")   // "orders" is the logical service name in Eureka
interface OrdersClient {
    @GetMapping("/orders/{id}")
    Order get(@PathVariable String id);
}
```

Under the hood: `Feign` → Spring Cloud LoadBalancer → cached list from Eureka → HTTP to a pod.

**Trade-offs you'll hit:**
- Registry outages are survived because every client has its own cache.
- But that same cache is *eventually consistent* — you can briefly talk to an instance that
  just crashed, until the next refresh cycle. That's why clients also need **retries** and
  **circuit breakers** (see the `circuit-breaker` lab next door).


## 4. Service Meshes (Istio, Linkerd, Envoy, Consul Connect)

In a **service mesh**, every service gets a **sidecar proxy** (usually Envoy) running next
to it. The sidecar handles discovery, load balancing, retries, mTLS, and observability, so
the application makes plain `localhost` calls and the mesh takes care of the rest.

**Mental model**

```
orders-app ─localhost→ orders-sidecar ─mTLS→ users-sidecar ─localhost→ users-app
                             ▲
                             │ gets endpoints from the mesh control plane
                             │ (which watches Kubernetes / Consul / etc.)
```

- **Third-party registration** — the control plane watches the platform (K8s pods, VMs in
  Consul) and pushes endpoint updates to every sidecar.
- **Client-side-style load balancing** — the sidecar picks the instance, so we avoid the
  central-bottleneck problem of a big shared proxy.
- **Zero application code change** — your service just calls `http://users/` as before.

This is the dominant pattern for large, polyglot microservice fleets today.


## 🧪 CAP trade-off: AP vs CP registries

When the network splits, a registry has to pick: stay available (AP) or stay consistent (CP)?

- **Eureka — AP.** During a partition, Eureka servers keep serving whatever they last knew,
  even if it's stale. You can still *find* instances, you might just talk to a dead one.
  *Availability over freshness.*
- **Consul / etcd / ZooKeeper — CP.** They rely on quorum (Raft). If a server loses quorum,
  it refuses writes (and sometimes reads) rather than return stale data.
  *Consistency over availability.*
- **Kubernetes** uses **etcd (CP)** as its source of truth, but the data plane (kube-proxy,
  CoreDNS) keeps caches, so short control-plane outages don't stop running pods from talking.

**Rule of thumb**

- Are lookups on the hot path of every request? Prefer **AP + client-side cache** so
  a registry blip isn't an outage.
- Is this a strongly-consistent config store (leader election, locks)? Pick **CP**.

This pairs with the client-side caching you built in Notebook 2 — it's how AP systems keep
working through control-plane wobbles.


## 🚫 When you do *not* need service discovery

Discovery is infrastructure you have to run, secure, and page for. Skip it when:

- **The address is genuinely stable.** A managed database, an external SaaS API, a
  cloud load balancer's DNS name — these don't move. Config is the right answer.
- **You already have Kubernetes.** You have service discovery. Deploying Consul or
  Eureka *next to* a `Service` object gives you two registries that can disagree.
- **You have one instance of one thing.** Discovery solves elasticity. Without
  elasticity there's nothing to discover.
- **A message broker already decouples you.** Producers and consumers of a Kafka topic
  never need each other's addresses — the broker's address is the only one that matters,
  and it's stable.

And two warnings for when you *do* need it:

- **The registry is on the hot path of every request.** Cache aggressively, prefer an AP
  registry, and make absolutely sure a registry outage degrades rather than kills you.
- **A healthy lookup is not a healthy instance.** There is always a window between an
  instance dying and the registry noticing. Discovery narrows that window; only
  **retries** and a **circuit breaker** actually cover it.

## 🧭 Which one should I pick?

- **Deploying to Kubernetes?** Just use Services + DNS. Add a mesh only when you need mTLS,
  traffic shifting, or fine-grained retries across many languages.
- **VMs / mixed environment?** Consul is the pragmatic default — it works everywhere and
  has a sidecar story (`consul-template`, Consul Connect) when you grow into it.
- **Spring Boot shop on VMs?** Eureka + Spring Cloud LoadBalancer is the path of least resistance.
- **Polyglot, want mTLS + traffic policy?** Adopt a service mesh (Istio / Linkerd).

## 🔗 See also in this repo

- `05-microservices/api-gateway/` — another piece of the ingress puzzle.
- `05-microservices/circuit-breaker/` — what you pair service discovery with to survive
  the moment *between* an instance dying and the registry noticing.
- `05-microservices/sidecar/` — the pattern meshes are built on.

## 📚 Further reading

- Chris Richardson, *Microservices Patterns*, chapter on service discovery.
- Kubernetes docs: *Services, Load Balancing, and Networking*.
- HashiCorp Consul Learn: *Service Discovery and Health Checks*.
- Netflix Tech Blog: *Eureka at Netflix*.
